In [2]:
from pathlib import Path
import http.client
import json
import os

from magic_pdf.data.data_reader_writer import FileBasedDataWriter, FileBasedDataReader
from magic_pdf.data.dataset import PymuDocDataset
from magic_pdf.model.doc_analyze_by_custom_model import doc_analyze
from magic_pdf.config.enums import SupportedPdfParseMethod

import toml

# from IPython.display import display, Markdown
# display(Markdown(cleaning_prompt))
# args
path_root = Path.cwd()
path_assets = path_root / 'assets'
pdf_file_name = "JFK00076.pdf"  # replace with the real pdf path
file_name, extension = pdf_file_name.split(".")
path_ouput = path_root / 'output' /f"{file_name}.md"

params = {
    "start_page_id" : 0,
    "end_page_id" : 5,
    "lang" : None,
    "formula_enable" : False,
    "table_enable" : False
}

# prepare env
local_image_dir, local_md_dir = "output/images", "output"
image_dir = str(os.path.basename(local_image_dir))

os.makedirs(local_image_dir, exist_ok=True)

image_writer, md_writer = FileBasedDataWriter(local_image_dir), FileBasedDataWriter(
    local_md_dir
)

# read bytes
reader1 = FileBasedDataReader("")
pdf_bytes = reader1.read(str(path_assets/file_name)+'.'+extension)  # read the pdf content

import tensorrt_llm failed, if do not use tensorrt, ignore this message
import lmdeploy failed, if do not use lmdeploy, ignore this message


In [3]:
# proc
## Create Dataset Instance
ds = PymuDocDataset(pdf_bytes)

try:
    config = toml.load("/workspace/MinerU/config.toml")

    paths = config['paths']
    path_output = paths['output_file']  # Get the output file path

    with open(path_output, "r") as f:
        file_ = f.read()

    with open(paths['prompt_cleaning'], "r") as f:
        cleaning_prompt = f.read()

    with open(paths['credentials'], "r") as f:
        credentials = json.load(f)

except FileNotFoundError as e:
    print("Error: One of the files did not exist, please, review.")
    raise e
except toml.TomlDecodeError:
    print("Error: Invalid TOML format in config.toml. Please check the file's syntax.")
except json.JSONDecodeError:
    print("Error: Invalid JSON format in credentials.json. Please check the file's syntax.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

2025-02-20 14:53:12.749 | INFO     | magic_pdf.data.dataset:__init__:156 - lang: None


In [4]:
ds??

Type:            PymuDocDataset
String form:     <magic_pdf.data.dataset.PymuDocDataset object at 0x7f1a81872470>
Length:          1
File:            /venv/main/lib/python3.10/site-packages/magic_pdf/data/dataset.py
Source:         
class PymuDocDataset(Dataset):
    def __init__(self, bits: bytes, lang=None):
        """Initialize the dataset, which wraps the pymudoc documents.

        Args:
            bits (bytes): the bytes of the pdf
        """
        self._raw_fitz = fitz.open('pdf', bits)
        self._records = [Doc(v) for v in self._raw_fitz]
        self._data_bits = bits
        self._raw_data = bits

        if lang == '':
            self._lang = None
        elif lang == 'auto':
            from magic_pdf.model.sub_modules.language_detection.utils import auto_detect_lang
            self._lang = auto_detect_lang(bits)
            logger.info(f"lang: {lang}, detect_lang: {self._lang}")
        else:
            self._lang = lang
            logger.info(f"lang: {lang}")


In [ ]:
## inference
infer_result = ds.apply(
    doc_analyze,
    ocr=ds.classify() == SupportedPdfParseMethod.OCR,
    **params
)

## pipeline
pipe_result = infer_result.pipe_ocr_mode(image_writer)

### get markdown content
md_content = pipe_result.get_markdown(image_dir)

### dump markdown
pipe_result.dump_md(md_writer, path_ouput, image_dir)

2025-02-20 14:51:45.031 | INFO     | magic_pdf.libs.pdf_check:detect_invalid_chars:57 - cid_count: 0, text_len: 711, cid_chars_radio: 0.0
2025-02-20 14:51:45.033 | WARNING  | magic_pdf.filter.pdf_classify_by_type:classify:334 - pdf is not classified by area and text_len, by_image_area: False, by_text: True, by_avg_words: True, by_img_num: True, by_text_layout: True, by_img_narrow_strips: True, by_invalid_chars: True
2025-02-20 14:51:45.037 | INFO     | magic_pdf.model.pdf_extract_kit:__init__:78 - DocAnalysis init, this may take some times, layout_model: doclayout_yolo, apply_formula: False, apply_ocr: True, apply_table: False, table_model: rapid_table, lang: None
2025-02-20 14:51:45.038 | INFO     | magic_pdf.model.pdf_extract_kit:__init__:99 - using device: cpu
2025-02-20 14:51:45.039 | INFO     | magic_pdf.model.pdf_extract_kit:__init__:103 - using models_dir: /root/.cache/huggingface/hub/models--opendatalab--PDF-Extract-Kit-1.0/snapshots/60416a2cabad3f7b7284b43ce37a99864484fba2/mod

**Prompt for Cleaning Extracted Markdown Files from Scanned PDFs:**

In [ ]:
conn = http.client.HTTPSConnection("generativelanguage.googleapis.com")
payload = json.dumps({
  "contents": [
    {
      "parts": [
        {
          "text": cleaning_prompt + file_
        }
      ]
    }
  ]
})
headers = {
  'Content-Type': 'application/json'
}


conn.request("POST", f"/v1beta/models/gemini-1.5-flash:generateContent?key={credentials["GEMINI_API_KEY"]}", payload, headers)
res = conn.getresponse()
data = res.read()

# Assuming `data` is a JSON string and `name_without_suff` is defined
clean_text = (
    json.loads(data.decode("utf-8"))
      .get("candidates", [{}])[0]
      .get("content", {})
      .get("parts", [{}])[0]
      .get("text", "")
     )

# Ensure output directory exists
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

# Define the output file path
output_file_path = os.path.join(output_dir, path_ouput)

# Write the cleaned text to the file
with open(output_file_path, "w", encoding="utf-8") as f:
    f.write(clean_text)

print(f"Cleaned text saved to: {output_file_path}")


from openai import OpenAI

client = OpenAI(api_key = credentials["OPENAI_API_KEY"])
speech_file_path = path_root / 'output' / "speech.opus"
response = client.audio.speech.create(
    model="tts-1",
    voice="alloy",
    input="Today is a wonderful day to build something people love!",
)
response.stream_to_file(speech_file_path)